# Kapitel 01 – Pods, Init-Container und Sidecars

Dieses Notebook begleitet die Beispiele aus `kubernetes/01-pods`. Jede Codezelle enthält **Bash-Befehle** (`%%bash`), die direkt auf dem K3s-Cluster unserer Trainingsumgebung ausgeführt werden. Führe die Zellen **der Reihe nach** aus.

**Voraussetzungen:** Jupyter läuft auf der Ubuntu-Trainings-VM, `kubectl` ist eingerichtet, und das Notebook liegt zusammen mit den vier YAML-Dateien im Verzeichnis `kubernetes/01-pods`. Die Befehle verwenden deshalb Dateinamen ohne Verzeichnispfad. Der Kernel muss in diesem Verzeichnis starten. Alle Übungen verwenden den Namespace `default`.

**Hinweis:** Dieses Notebook ist ausführbares Übungsmaterial. Die Kommandos verändern Ressourcen im Trainingscluster.

## Vorbereitung – Clusterverbindung prüfen

Zunächst prüfen wir, mit welchem Kubernetes-Cluster `kubectl` verbunden ist und ob die drei K3s-Nodes erreichbar sind.

In [1]:
%%bash
pwd
kubectl config current-context
kubectl get nodes -o wide


/home/sl01/org.javacream.training.kubernetes.basics/kubernetes/01-pods
default
NAME          STATUS   ROLES                  AGE   VERSION        INTERNAL-IP   EXTERNAL-IP   OS-IMAGE             KERNEL-VERSION      CONTAINER-RUNTIME
k3s-agent-1   Ready    <none>                 28h   v1.33.6+k3s1   172.18.0.4    <none>        Ubuntu 24.04.5 LTS   6.11.0-26-generic   containerd://2.1.5-k3s1.33
k3s-agent-2   Ready    <none>                 28h   v1.33.6+k3s1   172.18.0.5    <none>        Ubuntu 24.04.5 LTS   6.11.0-26-generic   containerd://2.1.5-k3s1.33
k3s-server    Ready    control-plane,master   28h   v1.33.6+k3s1   172.18.0.3    <none>        Ubuntu 24.04.5 LTS   6.11.0-26-generic   containerd://2.1.5-k3s1.33


Für die folgenden Übungen verwenden wir ausdrücklich den Namespace `default`. Dadurch vermeiden wir, dass ein zuvor gewählter Namespace das Ergebnis verändert. Die Einstellung gilt für den aktiven `kubectl`-Kontext.

In [2]:
%%bash
kubectl config set-context --current --namespace=default
kubectl get pods


Context "default" modified.
NAME                     READY   STATUS    RESTARTS   AGE
nginx-5654587fb9-b6h5x   1/1     Running   0          28h


## 1. `hello.yaml` – Einen Pod erstellen

**Lernziel:** Ein Pod kapselt einen oder mehrere Container. In `hello.yaml` definieren wir den Pod `hello-pod` mit einem Container, Labels, Port 8080 und Ressourcenlimits. Anders als bei einem Deployment verwalten wir diesen Pod direkt.

### 1.1 Manifest ansehen

Bevor wir etwas ausbringen, lesen wir die deklarative Definition.

In [3]:
%%bash
cat hello.yaml


apiVersion: v1
kind: Pod
metadata:
  name: hello-pod
  labels:
    zone: prod
    version: v1
spec:
  containers:
  - name: hello-ctr
    image: nigelpoulton/k8sbook:1.0
    ports:
    - containerPort: 8080
    resources:
      limits:
        memory: 256Mi
        cpu: "0.5"


### 1.2 Pod bereitstellen

`kubectl apply` überträgt die gewünschte Definition an den API-Server. Anschließend warten wir, bis der Pod bereit ist.

In [4]:
%%bash
kubectl apply -f hello.yaml
kubectl wait --for=condition=Ready pod/hello-pod --timeout=120s


pod/hello-pod created
pod/hello-pod condition met


### 1.3 Pod und Node untersuchen

Mit `-o wide` erkennen wir unter anderem die Pod-IP und den Node, auf dem der Scheduler den Pod platziert hat.

In [5]:
%%bash
kubectl get pod hello-pod -o wide


NAME        READY   STATUS    RESTARTS   AGE   IP           NODE          NOMINATED NODE   READINESS GATES
hello-pod   1/1     Running   0          21s   10.42.2.19   k3s-agent-1   <none>           <none>


### 1.4 Labels untersuchen

Die Labels `zone=prod` und `version=v1` stehen in den Metadaten. Über Selektoren können wir Pods gezielt filtern.

In [6]:
%%bash
kubectl get pods --show-labels
kubectl get pods -l zone=prod


NAME                     READY   STATUS    RESTARTS   AGE   LABELS
hello-pod                1/1     Running   0          41s   version=v1,zone=prod
nginx-5654587fb9-b6h5x   1/1     Running   0          28h   app=nginx,pod-template-hash=5654587fb9
NAME        READY   STATUS    RESTARTS   AGE
hello-pod   1/1     Running   0          41s


### 1.5 Containerkonfiguration und Ereignisse untersuchen

`describe` zeigt Image, Containerstatus, Limits und Events. Die vollständige Ressourcendefinition einschließlich serverseitig ergänzter Felder liefert `-o yaml`.

In [ ]:
%%bash
kubectl describe pod hello-pod


In [ ]:
%%bash
kubectl get pod hello-pod -o yaml


### 1.6 Logs ansehen

Die Standardausgabe des Containers wird über `kubectl logs` bereitgestellt.

In [7]:
%%bash
kubectl logs hello-pod --tail=30


### 1.7 HTTP-Anwendung testen

Für den Zugriff auf den Container-Port verwenden wir ein **kurzlebiges Port-Forwarding im Hintergrund**. Der Trap beendet den Hintergrundprozess automatisch, sobald die Bash-Zelle endet. So blockiert die Notebook-Zelle nicht. Die lokale Portnummer ist 18080, damit ein bereits belegter Port 8080 nicht stört.

In [8]:
%%bash
set -e
kubectl port-forward pod/hello-pod 18080:8080 >/tmp/k8s-hello-portforward.log 2>&1 &
pid=$!
trap 'kill "$pid" 2>/dev/null || true' EXIT
sleep 3
curl --max-time 10 -i http://127.0.0.1:18080/


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   599  100   599    0     0   1339      0 --:--:-- --:--:-- --:--:--  1343


HTTP/1.1 200 OK
X-Powered-By: Express
Content-Type: text/html; charset=utf-8
Content-Length: 599
ETag: W/"257-EeMQwo7NEwp2JE1clGdKSbHvuLQ"
Date: Tue, 24 Nov 2026 18:55:13 GMT
Connection: keep-alive
Keep-Alive: timeout=5

<html><head><title>K8s rocks</title><link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" integrity="sha384-QWTKZyjpPEjISv5WaRU9OFeRpok6YctnYmDr5pNlyT2bRjXh0JMhjY6hW+ALEwIH" crossorigin="anonymous"/></head><body><div class="container"><div class="jumbotron"><h1 style="text-align:center;">Kubernetes rocks!</h1><p style="text-align:center;">Check out my AI book.</p><p style="text-align:center;">          <a class="btn btn-primary" href="https://www.amazon.com/dp/1916585388">AI Explained: Facts, Fiction, and Future</a></p></div></div></body></html>

### 1.8 Aufräumen

Wir löschen den Pod. Da er nicht von einem Deployment verwaltet wird, wird er nicht automatisch neu erstellt.

In [9]:
%%bash
kubectl delete -f hello.yaml
kubectl get pods


pod "hello-pod" deleted
NAME                     READY   STATUS    RESTARTS   AGE
nginx-5654587fb9-b6h5x   1/1     Running   0          28h


## 2. `initpod.yml` und `initsvc.yml` – Startabhängigkeiten

**Lernziel:** Ein regulärer Init-Container muss erfolgreich beendet werden, bevor die Anwendungscontainer starten. Unser Init-Container fragt wiederholt den DNS-Namen `k8sbook` ab. Den Service erzeugen wir **absichtlich erst später**.

**Wichtig:** Hier wird nur die DNS-Auflösung geprüft, nicht die tatsächliche Erreichbarkeit eines Backend-Dienstes. Der Service selektiert in diesem vereinfachten Beispiel sogar den eigenen Pod.

### 2.1 Die beiden Manifeste untersuchen

`initpod.yml` definiert Init- und Hauptcontainer; `initsvc.yml` definiert den zunächst fehlenden Service.

In [ ]:
%%bash
cat initpod.yml
printf '\n--- SERVICE ---\n'
cat initsvc.yml


### 2.2 Zunächst nur den Pod erstellen

Der Init-Container startet, findet aber den Service-Namen noch nicht. Wir erwarten einen Initialisierungszustand wie `Init:0/1`.

In [ ]:
%%bash
kubectl apply -f initpod.yml
sleep 5
kubectl get pod initpod


### 2.3 Ursache untersuchen

In den Logs des Init-Containers erscheinen wiederholte DNS-Anfragen und Warteausgaben. Der Hauptcontainer ist noch nicht gestartet.

In [ ]:
%%bash
kubectl logs initpod -c init-ctr --tail=20


In [ ]:
%%bash
kubectl describe pod initpod


### 2.4 Fehlenden Service bestätigen

Der nächste Befehl soll `NotFound` melden. Wir behandeln dies ausdrücklich als erwartetes Übungsergebnis.

In [ ]:
%%bash
if kubectl get service k8sbook; then
  echo 'Der Service existiert bereits; die Ausgangssituation der Übung ist nicht gegeben.'
else
  echo 'Erwartet: Service k8sbook ist noch nicht vorhanden.'
fi


### 2.5 Service nachträglich bereitstellen

Jetzt erzeugen wir den Service. Kubernetes legt damit den DNS-Namen `k8sbook` an; der Init-Container kann seine Schleife verlassen.

In [ ]:
%%bash
kubectl apply -f initsvc.yml
kubectl get service k8sbook
kubectl wait --for=condition=Ready pod/initpod --timeout=120s


### 2.6 Zustandswechsel nachweisen

In den Logs steht nun `Service found!`. Der Init-Container ist beendet und der Hauptcontainer läuft.

In [ ]:
%%bash
kubectl logs initpod -c init-ctr
kubectl get pod initpod


In [ ]:
%%bash
kubectl get pod initpod -o jsonpath='{.status.initContainerStatuses[0].state.terminated.reason}'; echo
kubectl logs initpod -c web-ctr --tail=20


### 2.7 Webcontainer testen

Wir leiten den Container-Port 8080 kurzzeitig auf den lokalen Port 18080 weiter.

In [ ]:
%%bash
set -e
kubectl port-forward pod/initpod 18080:8080 >/tmp/k8s-initpod-portforward.log 2>&1 &
pid=$!
trap 'kill "$pid" 2>/dev/null || true' EXIT
sleep 3
curl --max-time 10 -i http://127.0.0.1:18080/


### 2.8 Aufräumen

Beide Ressourcen gehören zu dieser Übung und werden wieder entfernt.

In [ ]:
%%bash
kubectl delete -f initpod.yml
kubectl delete -f initsvc.yml


## 3. `initsidecar.yml` – GitSync und NGINX

**Lernziel:** Ein nativer Sidecar ist ein Init-Container mit `restartPolicy: Always`, der während der Lebensdauer des Pods weiterläuft. `ctr-sync` synchronisiert den Git-Branch `sidecar-example` in ein gemeinsames `emptyDir`-Volume. `ctr-web` (NGINX) liefert die synchronisierten Dateien aus. Das Manifest erstellt zusätzlich den Service `svc-sidecar`.

**Voraussetzung:** Der Cluster kann `registry.k8s.io` und GitHub erreichen. Das Beispiel verwendet GitSync 4.8.0. Der Branch `sidecar-example` enthält zunächst eine `index.html` mit Version **1.0**.

### 3.1 Manifest untersuchen

Achte auf `initContainers`, `restartPolicy: Always`, die GitSync-Umgebungsvariablen und das von beiden Containern verwendete Volume.

In [ ]:
%%bash
cat initsidecar.yml


### 3.2 Pod und Service bereitstellen

Ein einziger `apply`-Aufruf erzeugt beide Ressourcen aus dem mehrteiligen YAML-Dokument.

In [ ]:
%%bash
kubectl apply -f initsidecar.yml
kubectl wait --for=condition=Ready pod/git-sync --timeout=180s


### 3.3 Pod und Service untersuchen

Der Service ist vom Typ `LoadBalancer`. In unserer lokalen Trainingsumgebung muss dafür keine externe IP verfügbar sein; wir verwenden später Port-Forwarding.

In [ ]:
%%bash
kubectl get pod git-sync -o wide
kubectl get service svc-sidecar
kubectl describe pod git-sync


### 3.4 Logs der beiden Container vergleichen

Der Sidecar meldet Synchronisierungsvorgänge, NGINX protokolliert HTTP-Anfragen. Beide Container gehören zum selben Pod.

In [ ]:
%%bash
kubectl logs git-sync -c ctr-sync --tail=30
kubectl logs git-sync -c ctr-web --tail=20


### 3.5 Gemeinsames Volume untersuchen

GitSync legt unter `/tmp/git` einen symbolischen Link `html` auf den synchronisierten Checkout an. NGINX sieht dasselbe Volume unter `/usr/share/nginx/`. Die HTML-Datei ist daher über `/html/index.html` erreichbar.

In [ ]:
%%bash
kubectl exec git-sync -c ctr-web -- ls -la /usr/share/nginx/
kubectl exec git-sync -c ctr-web -- ls -la /usr/share/nginx/html/
kubectl exec git-sync -c ctr-web -- cat /usr/share/nginx/html/index.html


### 3.6 Anwendung über den Service aufrufen

Wir testen ausdrücklich den **Service** statt direkt den Pod. Die lokale Weiterleitung wird am Ende der Zelle wieder beendet.

In [ ]:
%%bash
set -e
kubectl port-forward service/svc-sidecar 18080:8080 >/tmp/k8s-sidecar-portforward.log 2>&1 &
pid=$!
trap 'kill "$pid" 2>/dev/null || true' EXIT
sleep 3
curl --max-time 10 -i http://127.0.0.1:18080/html/index.html


### 3.7 Änderung ohne Pod-Neustart demonstrieren

Öffne auf GitHub die Datei [index.html im Branch sidecar-example](https://github.com/Javacream/org.javacream.training.kubernetes.basics/blob/sidecar-example/index.html). Ändere **Version 1.0** auf **Version 1.1** und committe die Änderung. **Den Pod nicht neu starten!**

GitSync überprüft das Repository regelmäßig. Die folgenden Zellen zeigen, ob die neue Version übernommen wurde.

In [ ]:
%%bash
kubectl logs git-sync -c ctr-sync --tail=30
kubectl exec git-sync -c ctr-web -- cat /usr/share/nginx/html/index.html


Prüfe jetzt die HTTP-Antwort erneut. Falls noch Version 1.0 erscheint, warte kurz und führe die beiden Kontrollzellen erneut aus.

In [ ]:
%%bash
set -e
kubectl port-forward service/svc-sidecar 18080:8080 >/tmp/k8s-sidecar-portforward.log 2>&1 &
pid=$!
trap 'kill "$pid" 2>/dev/null || true' EXIT
sleep 3
curl --max-time 10 -i http://127.0.0.1:18080/html/index.html


### 3.8 Aufräumen

Das Manifest entfernt Pod **und** Service. Der Commit im Branch `sidecar-example` bleibt bestehen; vor einer erneuten Durchführung kann die HTML-Version auf 1.0 zurückgesetzt werden.

In [ ]:
%%bash
kubectl delete -f initsidecar.yml


## Abschlusskontrolle

Sind alle Übungsressourcen entfernt? Der standardmäßige Kubernetes-Service `kubernetes` bleibt erhalten.

In [ ]:
%%bash
kubectl get pods
kubectl get services
